In [5]:
from kaggle_secrets import UserSecretsClient
secrets      = UserSecretsClient()
OPENAI_KEY   = secrets.get_secret("OPENAI_API_KEY")
PINECONE_KEY = secrets.get_secret("PINECONE_API_KEY")

print("✓ Keys loaded")
print(f"  OpenAI  key: {OPENAI_KEY[:8]}...")
print(f"  Pinecone key: {PINECONE_KEY[:8]}...")

✓ Keys loaded
  OpenAI  key: sk-proj-...
  Pinecone key: pcsk_5h7...


In [6]:
# ============================================================
# Embed ALL 3 chunking strategies and upsert to Pinecone
# ============================================================

import subprocess
subprocess.run(["pip", "install", "-q", "openai", "pinecone"], check=True)


import os
import pickle
import time
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
from kaggle_secrets import UserSecretsClient

# ── API Keys from Kaggle Secrets ──────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("OPENAI_API_KEY")
secret_value_1 = user_secrets.get_secret("PINECONE_API_KEY")

# ── Config ────────────────────────────────────────────────────────────────────
EMBED_MODEL  = "text-embedding-3-small"
DIMENSION    = 1536
BATCH_SIZE   = 100
WORKING_DIR  = "/kaggle/input/datasets/rockybhai159/optimus"

# Each corpus pkl maps to its own Pinecone index
CORPORA = [
    {
        "pkl":        "corpus_A_fixed.pkl",
        "index_name": "rag-index-a-fixed",
        "label":      "Strategy A — Fixed",
    },
    {
        "pkl":        "corpus_B_recursive.pkl",
        "index_name": "rag-index-b-recursive",
        "label":      "Strategy B — Recursive  ← PRODUCTION",
    },
    {
        "pkl":        "corpus_C_semantic.pkl",
        "index_name": "rag-index-c-semantic",
        "label":      "Strategy C — Semantic",
    },
]

# ── Clients ───────────────────────────────────────────────────────────────────
openai_client = OpenAI(api_key=OPENAI_KEY)
pc            = Pinecone(api_key=PINECONE_KEY)

# ── Helper: embed a list of texts ─────────────────────────────────────────────
def embed_chunks(chunks):
    all_embeddings = []
    for i in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[i : i + BATCH_SIZE]
        batch_clean = [c.replace("\n", " ").strip() for c in batch]
        batch_clean = [c if c else "empty" for c in batch_clean]

        response = openai_client.embeddings.create(
            model=EMBED_MODEL,
            input=batch_clean,
        )
        all_embeddings.extend([item.embedding for item in response.data])
        print(f"    Embedded {min(i+BATCH_SIZE, len(chunks))}/{len(chunks)}")

    return all_embeddings

# ── Helper: create Pinecone index ────────────────────────────────────────────
def get_or_create_index(index_name):
    existing = [i.name for i in pc.list_indexes()]

    if index_name in existing:
        existing_dim = pc.describe_index(index_name).dimension
        if existing_dim != DIMENSION:
            print(f"  Wrong dimension ({existing_dim}), deleting and recreating...")
            pc.delete_index(index_name)
            time.sleep(5)
            existing = []

    if index_name not in [i.name for i in pc.list_indexes()]:
        print(f"  Creating index '{index_name}'...")
        pc.create_index(
            name=index_name,
            dimension=DIMENSION,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
        time.sleep(5)
    else:
        print(f"  Index '{index_name}' already exists")

    return pc.Index(index_name)

# ── Helper: upsert to Pinecone ────────────────────────────────────────────────
def upsert_to_pinecone(index, chunks, metadata, embeddings):
    for i in range(0, len(chunks), BATCH_SIZE):
        batch_end    = min(i + BATCH_SIZE, len(chunks))
        batch_chunks = chunks[i:batch_end]
        batch_embeds = embeddings[i:batch_end]
        batch_meta   = metadata[i:batch_end]

        vectors = []
        for j in range(len(batch_chunks)):
            m = batch_meta[j]
            vectors.append((
                f"chunk-{i+j}",
                batch_embeds[j],
                {
                    "text":     batch_chunks[j],
                    "source":   str(m["source"]),
                    "title":    str(m["title"]),
                    "authors":  str(m["authors"]),
                    "year":     str(m["year"]),
                    "category": str(m["category"]),
                    "chunk_id": int(m["chunk_id"]),
                }
            ))

        index.upsert(vectors=vectors)
        print(f"    Upserted {batch_end}/{len(chunks)}")

# ── Main loop — process all 3 corpora ─────────────────────────────────────────
summary = {}

for corpus in CORPORA:
    pkl_path = os.path.join(WORKING_DIR, corpus["pkl"])

    print(f"\n{'='*60}")
    print(f"  {corpus['label']}")
    print(f"{'='*60}")

    # Load corpus
    print(f"  Loading {corpus['pkl']}...")
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    chunks   = data["chunks"]
    metadata = data["metadata"]
    print(f"  Loaded {len(chunks)} chunks")

    # Estimate cost
    total_tokens = sum(len(c.split()) * 1.3 for c in chunks)
    est_cost     = total_tokens / 1_000_000 * 0.02
    print(f"  Estimated cost: ~${est_cost:.4f} USD")

    # Embed
    print(f"  Generating embeddings...")
    t0         = time.time()
    embeddings = embed_chunks(chunks)
    embed_time = round(time.time() - t0, 1)
    print(f"  ✓ Embeddings done in {embed_time}s")

    # Pinecone
    print(f"  Setting up Pinecone index...")
    index = get_or_create_index(corpus["index_name"])

    print(f"  Upserting to Pinecone...")
    t0 = time.time()
    upsert_to_pinecone(index, chunks, metadata, embeddings)
    upsert_time = round(time.time() - t0, 1)

    # Verify
    time.sleep(3)
    stats = index.describe_index_stats()

    summary[corpus["label"]] = {
        "chunks":       len(chunks),
        "vectors":      stats["total_vector_count"],
        "index":        corpus["index_name"],
        "embed_time":   embed_time,
        "upsert_time":  upsert_time,
    }

    print(f"  ✓ Done — {stats['total_vector_count']} vectors in '{corpus['index_name']}'")

# ── Final summary ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  EMBEDDING SUMMARY")
print(f"{'='*60}")
print(f"  {'Strategy':<35} {'Chunks':>7}  {'Vectors':>8}  {'Time':>8}")
print(f"  {'-'*60}")

for label, s in summary.items():
    status = "✓" if s["chunks"] == s["vectors"] else "⚠"
    print(f"  {status} {label:<33} {s['chunks']:>7}  {s['vectors']:>8}  {s['embed_time']:>6}s")

print(f"\n  Pinecone indexes created:")
for label, s in summary.items():
    print(f"    {s['index']}")

print(f"\n  Production index  →  rag-index-b-recursive")
print(f"  Ablation indexes  →  rag-index-a-fixed, rag-index-c-semantic")
print(f"{'='*60}")


  Strategy A — Fixed
  Loading corpus_A_fixed.pkl...
  Loaded 2007 chunks
  Estimated cost: ~$0.0037 USD
  Generating embeddings...
    Embedded 100/2007
    Embedded 200/2007
    Embedded 300/2007
    Embedded 400/2007
    Embedded 500/2007
    Embedded 600/2007
    Embedded 700/2007
    Embedded 800/2007
    Embedded 900/2007
    Embedded 1000/2007
    Embedded 1100/2007
    Embedded 1200/2007
    Embedded 1300/2007
    Embedded 1400/2007
    Embedded 1500/2007
    Embedded 1600/2007
    Embedded 1700/2007
    Embedded 1800/2007
    Embedded 1900/2007
    Embedded 2000/2007
    Embedded 2007/2007
  ✓ Embeddings done in 19.6s
  Setting up Pinecone index...
  Creating index 'rag-index-a-fixed'...
  Upserting to Pinecone...
    Upserted 100/2007
    Upserted 200/2007
    Upserted 300/2007
    Upserted 400/2007
    Upserted 500/2007
    Upserted 600/2007
    Upserted 700/2007
    Upserted 800/2007
    Upserted 900/2007
    Upserted 1000/2007
    Upserted 1100/2007
    Upserted 1200/2007

In [7]:
from pinecone import Pinecone

secret_value_1 = user_secrets.get_secret("PINECONE_API_KEY")
# Initialize Pinecone (use your API key)
pc = Pinecone(api_key=PINECONE_KEY)

# Connect to your index
index = pc.Index("rag-index-a-fixed")

# Get stats
stats = index.describe_index_stats()

# Print full stats
print(stats)

# Print total vector count only
print("Total vectors:", stats["total_vector_count"])

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '188',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 04 Apr 2026 03:36:07 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '4',
                                    'x-pinecone-request-latency-ms': '3',
                                    'x-pinecone-response-duration-ms': '5'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 2007}},
 'storageFullness': 0.0,
 'total_vector_count': 2007,
 'vector_type': 'dense'}
Total vectors: 2007


In [8]:
from pinecone import Pinecone

secret_value_1 = user_secrets.get_secret("PINECONE_API_KEY")
# Initialize Pinecone (use your API key)
pc = Pinecone(api_key=PINECONE_KEY)

# Connect to your index
index = pc.Index("rag-index-b-recursive")

# Get stats
stats = index.describe_index_stats()

# Print full stats
print(stats)

# Print total vector count only
print("Total vectors:", stats["total_vector_count"])

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '188',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 04 Apr 2026 03:37:20 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '3',
                                    'x-pinecone-request-latency-ms': '2',
                                    'x-pinecone-response-duration-ms': '4'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 2753}},
 'storageFullness': 0.0,
 'total_vector_count': 2753,
 'vector_type': 'dense'}
Total vectors: 2753


In [9]:
from pinecone import Pinecone

secret_value_1 = user_secrets.get_secret("PINECONE_API_KEY")
# Initialize Pinecone (use your API key)
pc = Pinecone(api_key=PINECONE_KEY)

# Connect to your index
index = pc.Index("rag-index-c-semantic")

# Get stats
stats = index.describe_index_stats()

# Print full stats
print(stats)

# Print total vector count only
print("Total vectors:", stats["total_vector_count"])

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '188',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 04 Apr 2026 03:38:19 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '3',
                                    'x-pinecone-request-latency-ms': '2',
                                    'x-pinecone-response-duration-ms': '4'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 1455}},
 'storageFullness': 0.0,
 'total_vector_count': 1455,
 'vector_type': 'dense'}
Total vectors: 1455
